In [5]:
import os
from pyspark.sql import SparkSession
import hashlib


source_filepath = '../data/incoming/yellow_tripdata_2026-04.parquet'


def calculate_file_sha256(file_path: str) -> str:
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(65536), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()
computed_checksum = calculate_file_sha256(source_filepath)
print(f"🔒 File checksum: {computed_checksum}")

# e03b82d5722a6b21d4fbce011afd8837495202c0b5db77ff0d8d15dd80834ba8
# e03b82d5722a6b21d4fbce011afd8837495202c0b5db77ff0d8d15dd80834ba8

🔒 File checksum: e03b82d5722a6b21d4fbce011afd8837495202c0b5db77ff0d8d15dd80834ba8


In [9]:
from pyspark.sql.functions import lit, current_timestamp, col

spark = (SparkSession.builder
    .appName("MedallionPlatformAuditor")
    .config("spark.sql.catalog.nyc", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nyc.type", "hadoop")
    .config("spark.sql.catalog.nyc.warehouse", "/opt/airflow/data/warehouse")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .getOrCreate())

   
ledger_table = f"{catalog}.audit.ingestion_ledger"

dup_ledger = spark.table(ledger_table).where((col("file_checksum") == computed_checksum) & (col("status") == "SUCCESS")).count()
# if dup_ledger > 0:
#     print("Idempotency guard: checksum already recorded in ledger. Skipping.")
#     sys.exit(0)

In [12]:
spark.table(ledger_table).toPandas()

,source_filename,file_checksum,ingestion_run_id,status,finished_at
0,yellow_tripdata_2026-05.parquet,9aa5a1609e2bf07d9051b7d530de05b1019e12a560ecb2...,asset_triggered__2026-07-01T22:25:11.741237+00...,SUCCESS,2026-07-01 22:25:23.225281
1,yellow_tripdata_2026-04.parquet,e03b82d5722a6b21d4fbce011afd8837495202c0b5db77...,asset_triggered__2026-07-01T22:24:11.304874+00...,SUCCESS,2026-07-01 22:24:22.126396
